# Train unified and split policies
    Default: full tuning, paper-stated settings and documented shared assumptions.
    Both conditions run this codebase; our unified run is the comparison baseline.
    Run notebook01 first and use
    the same WORK/DATA location. The short pilot only reduces update count.
    Commands are printed until RUN_TRAINING is enabled; no GPU is provisioned.

    Unified shares parameters and optimizer moments. Split has separate parameters
    and moments; the inactive live model supplies opponent rollouts. Only unified
    needs a frozen opponent snapshot. Both use one global learning-rate schedule.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile

candidates = [Path(os.environ.get("GASP_ROOT", ".")), Path.cwd(), Path.cwd().parent, Path("/content/gasp")]
ROOT = next((p.resolve() for p in candidates if (p / "src/gasp").is_dir()), None)
if ROOT is None:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("Unzip dist/gasp-colab.zip and open this notebook inside the project.")
    print("Select the delivered gasp-colab.zip archive.")
    uploaded = files.upload()
    archives = [p for p in uploaded if p.endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Select exactly one project archive")
    with zipfile.ZipFile(archives[0]) as z:
        for name in z.namelist():
            if not name.startswith("gasp/") or ".." in Path(name).parts:
                raise ValueError("Unexpected archive member")
        z.extractall("/content")
    ROOT = Path("/content/gasp")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

def command(*args):
    result = subprocess.run([sys.executable, *map(str,args)], cwd=ROOT,
                            env={**os.environ,"PYTHONPATH":str(ROOT/"src")},
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    result.check_returncode()

print("Project:", ROOT)


In [ ]:
# Run once per new Colab runtime. Restart the runtime afterward, then rerun bootstrap.
INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    assert (3,11) <= sys.version_info[:2] < (3,14), "Use Python 3.11–3.13"
    # These optional image/audio packages can conflict with a pinned text-only torch.
    command("-m", "pip", "uninstall", "-y", "torchvision", "torchaudio")
    command("-m", "pip", "install", "-r", "requirements-colab.txt")
    command("-m", "pip", "install", "--no-deps", ".")
    print("Restart runtime if packages were already imported. No training has started.")


In [ ]:
# Optional persistence. Keep downloaded HF weights in Colab scratch; persist data/runs.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/gasp-experiment")
else:
    WORK = ROOT
WORK.mkdir(parents=True, exist_ok=True)
print("Data and outputs:", WORK)


In [ ]:
PROFILE = "full"
DATA = WORK/f"data/{PROFILE}-v4"
RUNS = WORK/f"runs/{PROFILE}-v4"
from gasp.config import Config
cfg = Config.load(DATA/"resolved_config.json")
print(cfg)


In [ ]:
import shlex
RUN_TRAINING = False
SEEDS = [42, 52, 62] if PROFILE == "full" else [42]
# Separate processes release GPU allocations and reinitialize each seed.
for seed in SEEDS:
    for condition in ("unified", "split"):
        out = RUNS/f"{condition}-{seed}"
        args=["-m", "gasp", "train", "--config", DATA/"resolved_config.json",
              "--condition", condition, "--seed", seed,
              "--train-cache", DATA/"train.jsonl", "--eval-cache", DATA/"eval.jsonl",
              "--output", out]
        if (out/"checkpoint.pt").exists():
            args += ["--resume", out/"checkpoint.pt"]
        print(shlex.join([sys.executable, *map(str,args)]))
        if RUN_TRAINING:
            command(*args)


In [ ]:
for seed in SEEDS:
    for condition in ("unified", "split"):
        out=RUNS/f"{condition}-{seed}"
        if not (out/"training.jsonl").exists():
            continue
        history=[json.loads(x) for x in (out/"training.jsonl").read_text().splitlines()]
        contexts=[c for h in history for c in h["contexts"]]
        print(condition, seed, "clean generation attempts:", sum(c["clean_attempts"] for c in contexts),
              "informative samples:", sum(c["informative_samples"] for c in contexts))
        print(json.dumps(json.loads((out/"telemetry.json").read_text()), indent=2))


Clean generation retries can cost different amounts in each condition.
    A failure to obtain a correct trace stops the run; investigate before resuming
    from the last checkpoint. No cached trace or synthetic corruption is substituted.
    Pre-fix checkpoints use a different protocol and cannot be resumed here.
